# 04 · LLM Model Internals

> **Focus**: VRAM calculations, quantization impact, inference profiling

This notebook explores the practical constraints of running LLMs:
1. **VRAM Calculator** — determine memory requirements for different model configs
2. **Quantization Benchmark** — measure accuracy/memory/speed trade-offs
3. **Inference Profiling** — identify bottlenecks in forward pass

**Requirements**: GPU with ≥8 GB VRAM, PyTorch, transformers, bitsandbytes

## 0 · Environment Setup

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

# Verify GPU availability
if not torch.cuda.is_available():
    print("⚠ Warning: No GPU detected — some exercises will be skipped")
    print("  Exercise 1 (VRAM calculator) can run on CPU")
    print("  Exercises 2-3 require GPU")
else:
    device = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU: {device}")
    print(f"  VRAM: {vram:.1f} GB")

## Exercise 0 — Parameter Breakdown

**Goal**: Understand where parameters are distributed in a transformer model.

**LLaMA 7B breakdown** (see chapter §6A):
- **Token embeddings**: 32k vocab × 4096 dim = 131M params
- **Per transformer block** (32 total):
  - Attention (Q, K, V, O): 4 × (4096 × 4096) = 67M params
  - FFN (W1, W2): (4096 × 11008) + (11008 × 4096) = 90M params
  - LayerNorms: ~16K params (negligible)
  - **Total per block**: 157M params
- **All 32 blocks**: 32 × 157M = 5.0B params
- **Output head**: 4096 × 32k = 131M params
- **Total**: 6.74B params (~7B advertised)

**Key insight**: ~75% of parameters are in FFN layers (the "heavy lifters").

In [ ]:
def count_model_parameters(
    vocab_size: int,
    d_model: int,
    num_layers: int,
    d_ff: int,
) -> dict:
    """
    Calculate parameter counts by component.

    Args:
        vocab_size: Vocabulary size (e.g., 32000 for LLaMA)
        d_model: Hidden dimension (e.g., 4096 for LLaMA 7B)
        num_layers: Number of transformer blocks (e.g., 32)
        d_ff: FFN intermediate dimension (e.g., 11008)

    Returns:
        dict with keys: embeddings, attention_per_block, ffn_per_block,
                        total_per_block, all_blocks, output_head, total
    """
    # TODO: Calculate parameter counts for each component
    # 1. Token embeddings: vocab_size × d_model
    # 2. Attention per block: 4 × (d_model × d_model)  [Q, K, V, O projections]
    # 3. FFN per block: (d_model × d_ff) + (d_ff × d_model)  [W1, W2]
    # 4. LayerNorms: negligible (2 × d_model × 2 for mean/var)
    # 5. Output head: d_model × vocab_size (often shares weights with embeddings)

    pass

In [ ]:
# TODO: Calculate parameter breakdown for LLaMA 13B and 70B
# LLaMA 13B config: vocab=32000, d_model=5120, num_layers=40, d_ff=13824
# LLaMA 70B config: vocab=32000, d_model=8192, num_layers=80, d_ff=28672
#
# Create a DataFrame comparing all three models
# Show: model_name, embeddings, attention_per_block, ffn_per_block,
#       total_per_block, all_blocks, output_head, total
# Calculate FFN percentage for each model

### Compare across model sizes

## Exercise 1 — VRAM Calculator

**Goal**: Calculate memory requirements for different model configurations.

**Memory components**:
- **Model weights**: `params × bytes_per_param`
- **Activations**: `batch_size × seq_len × hidden_dim × num_layers × bytes_per_activation`
- **KV Cache**: `2 × batch_size × seq_len × num_layers × num_heads × head_dim × bytes_per_param`

**Precision mapping**:
- fp32: 4 bytes
- fp16/bf16: 2 bytes
- int8: 1 byte
- int4: 0.5 bytes

In [ ]:
def calculate_vram(
    params: int,           # Total parameters (e.g., 7e9 for 7B)
    precision: str,        # 'fp32', 'fp16', 'int8', 'int4'
    seq_len: int,          # Sequence length
    batch_size: int,       # Batch size
    hidden_dim: int = 4096,      # Model hidden dimension
    num_layers: int = 32,        # Number of transformer layers
    num_heads: int = 32,         # Number of attention heads
) -> dict:
    """
    Calculate VRAM requirements for LLM inference.

    Returns:
        dict with keys: weights_gb, activations_gb, kv_cache_gb, total_gb
    """
    # TODO: Implement VRAM calculation
    # 1. Map precision to bytes per parameter
    # 2. Calculate weights memory
    # 3. Calculate activations memory (rough estimate)
    # 4. Calculate KV cache memory
    # 5. Return breakdown as dict

    pass

### Test on LLaMA configurations

In [ ]:
# TODO: Test calculator on LLaMA 7B, 13B, 70B
# Configuration reference:
# - LLaMA 7B: 7B params, 4096 hidden_dim, 32 layers, 32 heads
# - LLaMA 13B: 13B params, 5120 hidden_dim, 40 layers, 40 heads
# - LLaMA 70B: 70B params, 8192 hidden_dim, 80 layers, 64 heads
#
# Test with:
# - Precisions: fp16, int8, int4
# - Sequence lengths: 512, 2048
# - Batch size: 1
#
# Create a DataFrame showing results

### Can it fit on RTX 4090?

In [ ]:
# TODO: Determine which configurations fit on RTX 4090 (24 GB VRAM)
# Create a table showing: model, precision, seq_len, total_vram, fits_4090

## Exercise 2 — Quantization Impact Benchmark

**Goal**: Compare model quality, memory usage, and speed across quantization levels.

**Setup**: Load same model (GPT-2 or similar) in fp16, int8, int4.

**Note**: This exercise requires GPU. Skip if CUDA not available.

In [ ]:
if not torch.cuda.is_available():
    print("⚠ Skipping Exercise 2 — GPU required")
else:
    print("✓ GPU available — proceeding with quantization benchmark")

### Load model in different precisions

In [ ]:
# TODO: Load GPT-2 (or similar small model) in three configurations:
# 1. fp16 (half precision)
# 2. int8 (8-bit quantization using bitsandbytes)
# 3. int4 (4-bit quantization using bitsandbytes)
#
# For each:
# - Record VRAM usage after loading (torch.cuda.memory_allocated())
# - Store model reference
#
# Hints:
# - Use BitsAndBytesConfig from transformers for int8/int4
# - Load with load_in_8bit=True or load_in_4bit=True
# - Clear GPU memory between loads: torch.cuda.empty_cache()

### Run benchmark on sample prompts

In [ ]:
# TODO: Create benchmark prompts (5-10 diverse examples)
# For each model (fp16, int8, int4):
#   1. Generate outputs for all prompts
#   2. Measure inference time per prompt
#   3. Calculate perplexity or similar quality metric
#
# Store results in DataFrame with columns:
# - precision, prompt_id, generation_time_ms, output_quality_score

benchmark_prompts = [
    "The capital of France is",
    "To calculate the area of a circle, you need",
    "In 1969, humans landed on",
    "The speed of light is approximately",
    "Python is a programming language that",
]

# TODO: Implement benchmark loop

### Create comparison table

In [ ]:
# TODO: Aggregate results and create summary table
# Columns: precision, avg_time_ms, memory_gb, quality_score
# Calculate speedup relative to fp16
# Calculate memory savings relative to fp16

## Exercise 3 — Inference Profiling

**Goal**: Identify where inference time is spent in the forward pass.

**Method**: Use `torch.profiler` to break down time by operation type.

**Compare**: Attention vs FFN vs LayerNorm at different sequence lengths.

In [ ]:
if not torch.cuda.is_available():
    print("⚠ Skipping Exercise 3 — GPU required for profiling")
else:
    print("✓ GPU available — proceeding with inference profiling")

### Profile forward pass

In [ ]:
# TODO: Profile GPT-2 forward pass at different sequence lengths
# 1. Load GPT-2 model
# 2. For each seq_len in [128, 512, 2048]:
#    - Create input tokens of that length
#    - Profile forward pass using torch.profiler.profile()
#    - Extract time spent in: attention, FFN, layer norm
#
# Hints:
# - Use profile.key_averages() to get operation stats
# - Look for operations containing: 'matmul', 'attention', 'gelu', 'layer_norm'
# - Group by operation type

from torch.profiler import profile, ProfilerActivity

def profile_inference(model, tokenizer, seq_len: int) -> dict:
    """
    Profile model forward pass and return time breakdown.

    Returns:
        dict with keys: attention_ms, ffn_ms, layer_norm_ms, other_ms
    """
    # TODO: Implement profiling
    pass

### Compare across sequence lengths

In [ ]:
# TODO: Run profiling for seq_len in [128, 512, 2048]
# Create DataFrame with columns: seq_len, attention_ms, ffn_ms, layer_norm_ms, other_ms
# Calculate percentage of time in each operation

### Visualize time breakdown

In [ ]:
# TODO: Create stacked bar chart showing time breakdown by operation type
# X-axis: sequence length
# Y-axis: time (ms) or percentage
# Bars: stacked by operation (attention, FFN, layer_norm, other)
#
# Expected observation: attention time grows quadratically with seq_len

## Summary & Key Findings

**Exercise 0 — Parameter Breakdown**:
- Where parameters live: ?
- FFN percentage of each block: ?
- Total parameters calculated for LLaMA 7B: ?

**Exercise 1 — VRAM Calculator**:
- Key insight: ?
- Which models fit on RTX 4090: ?

**Exercise 2 — Quantization Impact**:
- Memory savings (int8 vs fp16): ?
- Speed improvement (int8 vs fp16): ?
- Quality degradation: ?

**Exercise 3 — Inference Profiling**:
- Dominant operation at short sequences: ?
- Dominant operation at long sequences: ?
- How does attention scale with seq_len: ?